In [1]:
%load_ext autoreload
%autoreload 2
%reset -f

In [2]:
from pathlib import Path
import os
from os.path import join
import sys

# Find KPIHub root (directory containing lib/tables), then import lib.* as a package.
_here = Path.cwd().resolve()
_root = next((p for p in [_here, *list(_here.parents)[:12]] if (p / "lib" / "tables").is_dir()), None)
if _root is None:
    raise FileNotFoundError(
        f"Could not find KPIHub root (folder containing lib/tables). cwd={Path.cwd()!r}"
    )
sys.path.insert(0, str(_root))
os.chdir(_root)

from locallib.picarrodb import *
from locallib.slack import *
from locallib.etl import Loggers
from locallib.pandas import *

from lib.tables.IngesterTables import *
from lib.handlers.CustomerHandler import *
from lib.config import *
from lib.KPIHubConnection import *
from lib.query.bank import *

from datetime import date
from datetime import timedelta

EU1_Conn created successfully
EU2_Conn created successfully
DataHub_Conn created successfully
US_Conn created successfully
EU1_PROD_Conn created successfully
EU2_PROD_Conn created successfully


In [3]:
customer_name = 'Cadent'
customer_id = Query(query = f"SELECT CustomerId FROM KPI_Customer WHERE Name = '{customer_name}'").execute([KPIHub_Conn]).values[0][0]

tableList = [KPI_ReportSummary,KPI_EmissionSourceSummary]
aggregator = ['BoundaryRegion', 'ReportYear', 'ReportWeek']
data = {}
for table in tableList:
    data[table] = Query(query = f"SELECT * FROM {table.name} WHERE ReportId IN (SELECT ReportId FROM KPI_ReportSummary WHERE CustomerId IN (SELECT CustomerId FROM KPI_Customer WHERE Name = '{customer_name}'))").execute([KPIHub_Conn])

denominator = 'DistributionPipeCoveredKm'

def quarter_density_share_apply(df):
    out = {}
    denom = df[denominator].sum()
    lisa_count = df["LisaCount"].sum()
    out["LisaCount"] = lisa_count
    out["EmissionRate"] = df["EmissionRate"].sum()
    out["B0Count"] = df["B0Count"].sum()
    out["B1Count"] = df["B1Count"].sum()
    out["Bm1Count"] = df["Bm1Count"].sum()
    out["Bm2Count"] = df["Bm2Count"].sum()
    out["NGCount"] = df["NGCount"].sum()
    out["PGCount"] = df["PGCount"].sum()
    out["Not_NGCount"] = df["Not_NGCount"].sum()

    out["LisaDensity"] = lisa_count / denom if denom else None
    out["InstatanoeusEmission"] = out["EmissionRate"] / denom if denom else None
    out["B0Density"] = out["B0Count"] / denom if denom else None
    out["B1Density"] = out["B1Count"] / denom if denom else None
    out["B-1Density"] = out["Bm1Count"] / denom if denom else None
    out["B-2Density"] = out["Bm2Count"] / denom if denom else None
    out["NGDensity"] = out["NGCount"] / denom if denom else None
    out["PGDensity"] = out["PGCount"] / denom if denom else None

    total_sum = out['NGCount'] + out['Not_NGCount'] + out['PGCount']

    out["B0Share"] = out["B0Count"] / lisa_count if lisa_count else None
    out["B1Share"] = out["B1Count"] / lisa_count if lisa_count else None
    out["B-1Share"] = out["Bm1Count"] / lisa_count if lisa_count else None
    out["B-2Share"] = out["Bm2Count"] / lisa_count if lisa_count else None
    out["NGShare"] = out["NGCount"] / total_sum if total_sum else None
    out["PGShare"] = out["PGCount"] / total_sum if total_sum else None
    out["Not_NGShare"] = out["Not_NGCount"] / total_sum if total_sum else None

    return pd.Series(out)

emission_by_report = pd.merge(data[KPI_ReportSummary],data[KPI_EmissionSourceSummary],on="ReportId",how="left")
emission_by_report_period= emission_by_report.groupby(aggregator).apply(quarter_density_share_apply)
emission_by_report_period = emission_by_report_period.round(2)

In [4]:
r = emission_by_report_period.reset_index()
r_long = r.melt(id_vars=aggregator, var_name='KPIId', value_name='Value')
r_long['Id'] = r_long.apply(lambda row: f"{row['KPIId']}_{customer_name}R{row['BoundaryRegion'].replace(' ','')}_Y{row['ReportYear']}_W{row['ReportWeek']}", axis=1)
r_long = r_long.rename(columns={'ReportWeek': 'PeriodValue', 'ReportYear': 'Year'})
r_long['PeriodType'] = "Weekly"
r_long['LastUpdated'] = datetime.now()
r_long['CustomerId'] = customer_id


In [5]:
KPI_Data.update_table(arguments = {'DataFrame': r_long, 'db_path': DB_PATH, 'PrimaryKey': 'Id'})
KPI_Data.query_table(arguments = {'db_path': DB_PATH})

,Id,KPIId,CustomerId,BoundaryRegion,Year,PeriodType,PeriodValue,Value,DataType,LastUpdated
0,FOVMain_Cadent_REastMidlands_Y2024_W16,FOVMain,BD4D080B-1D12-D329-ABD0-39FEB9804E98,East Midlands,2024,Weekly,16,None,None,2026-06-29 12:31:11.929485
1,FOVMain_Cadent_REastMidlands_Y2024_W35,FOVMain,BD4D080B-1D12-D329-ABD0-39FEB9804E98,East Midlands,2024,Weekly,35,0.95,None,2026-06-29 12:31:11.929485
2,FOVMain_Cadent_REastMidlands_Y2024_W36,FOVMain,BD4D080B-1D12-D329-ABD0-39FEB9804E98,East Midlands,2024,Weekly,36,0.96,None,2026-06-29 12:31:11.929485
3,FOVMain_Cadent_REastMidlands_Y2024_W48,FOVMain,BD4D080B-1D12-D329-ABD0-39FEB9804E98,East Midlands,2024,Weekly,48,0.94,None,2026-06-29 12:31:11.929485
4,FOVMain_Cadent_REastMidlands_Y2026_W5,FOVMain,BD4D080B-1D12-D329-ABD0-39FEB9804E98,East Midlands,2026,Weekly,5,0.97,None,2026-06-29 12:31:11.929485
...,...,...,...,...,...,...,...,...,...,...
21331,B-1Share_CadentRNorthLondon_Y2026_W27,B-1Share,BD4D080B-1D12-D329-ABD0-39FEB9804E98,North London,2026,Weekly,27,0.6,None,2026-06-30 12:37:28.099627
21332,B-2Share_CadentRNorthLondon_Y2026_W27,B-2Share,BD4D080B-1D12-D329-ABD0-39FEB9804E98,North London,2026,Weekly,27,0.12,None,2026-06-30 12:37:28.099627
21333,NGShare_CadentRNorthLondon_Y2026_W27,NGShare,BD4D080B-1D12-D329-ABD0-39FEB9804E98,North London,2026,Weekly,27,0.81,None,2026-06-30 12:37:28.099627
21334,PGShare_CadentRNorthLondon_Y2026_W27,PGShare,BD4D080B-1D12-D329-ABD0-39FEB9804E98,North London,2026,Weekly,27,0.14,None,2026-06-30 12:37:28.099627
